# Simple CorrDiff Ensemble Generation - FIXED

**Clean and minimal CorrDiff ensemble workflow with comprehensive analysis.**

## Key Features
- 🔧 **Simple Configuration**: All settings in one place
- 📊 **Explicit Grid Coordinates**: Uses hardcoded grids that work
- 🎯 **Ground Truth Support**: Includes ground truth in output if available
- 📈 **Comprehensive Analysis**: Multiple visualization and metrics plots
- 🧠 **Memory Efficient**: Proper GPU memory management

## Fixed Issues
- ✅ Grid coordinate inference issues resolved
- ✅ Uses explicit coordinates that match working 1.1 version
- ✅ Proper Earth2Studio coordinate validation
- ✅ Fixed dimension ordering in data loading
- ✅ Corrected coordinate handshaking between data source and model

In [1]:
# =============================================================================
# CONFIGURATION - FIXED TO MATCH WORKING 1.1 VERSION
# =============================================================================

VARIABLES = "Fog_index"

# Checkpoint paths
BASE_CHECKPOINTS_PATH = "/app/host/home/younes.abid/git/physicsnemo/outputs/checkpoints/"

REGRESSION_CHECKPOINT = BASE_CHECKPOINTS_PATH + VARIABLES + '/checkpoints_regression/UNet.0.390000.mdlus'
DIFFUSION_CHECKPOINT = BASE_CHECKPOINTS_PATH + VARIABLES + '/checkpoints_diffusion/EDMPrecondSuperResolution.0.590000.mdlus'

# Data paths
BASE_DATA_PATH = '/app/host/mnt/storage/younes.abid/physicsnemo/data/custom_data_2/'
DATA_FILE = BASE_DATA_PATH + 'ERA5_WRF_combined_concatenated_432/2024-04-30_2024-05-30_21.nc'
STATS_FILE = BASE_DATA_PATH + 'stats_432/stat.json'

# Variables
INPUT_VARIABLES = ['t_850', 't_500', 'z_850', 'z_500', 'u_850', 'u_500', 'v_850', 'v_500', 'u10', 'v10', 't2m', 'd2m', 'skt', 'sp', 'tcwv', 'tp']
OUTPUT_VARIABLES = ['Fog_index']

# FIXED: Use explicit grid coordinates (this is what makes 1.1 work)
# These coordinates MUST match your actual data domain
INPUT_GRID = {
    'lat': (19.0, 28.0, 432),   # (min, max, points)
    'lon': (116.0, 126.0, 432)  # (min, max, points)
}
OUTPUT_GRID = {
    'lat': (19.0, 28.0, 432),   # (min, max, points) 
    'lon': (116.0, 126.0, 432)  # (min, max, points)
}

# Ensemble parameters
INFERENCE_TIMES = ['2024-05-01T00:00:00', '2024-05-01T06:00:00', '2024-05-01T12:00:00']
NUM_ENSEMBLES = 8
SEED_BASE = 42
SAMPLING_MODE = 'stochastic'  # 'stochastic' or 'deterministic'
NUMBER_OF_STEPS = 20
SOLVER = 'euler'
HR_MEAN_CONDITIONING = True

# Output configuration
from datetime import datetime
TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_FILE = f'/app/outputs/generation/{VARIABLES}/{TIMESTAMP}/ensemble.nc'
PLOTS_DIR = f'/app/outputs/generation/{VARIABLES}/{TIMESTAMP}/plots'

# Verbose logging
VERBOSE = True

if VERBOSE:
    print('✅ Configuration loaded with EXPLICIT GRIDS')
    print(f'📊 Ensemble setup: {NUM_ENSEMBLES} members, {SAMPLING_MODE} sampling')
    print(f'🔄 Diffusion steps: {NUMBER_OF_STEPS}, solver: {SOLVER}')
    print(f'📐 Grid: {INPUT_GRID["lat"][2]}x{INPUT_GRID["lon"][2]} points')
    print(f'💾 Output: {OUTPUT_FILE}')

✅ Configuration loaded with EXPLICIT GRIDS
📊 Ensemble setup: 8 members, stochastic sampling
🔄 Diffusion steps: 20, solver: euler
📐 Grid: 432x432 points
💾 Output: /app/outputs/generation/Fog_index/20260203_134332/ensemble.nc


In [2]:
import os
import sys
import torch
import numpy as np
from pathlib import Path

# Add current directory to path for our modules
current_dir = Path.cwd() / 'src2'
if str(current_dir) not in sys.path:
    sys.path.insert(0, str(current_dir))

# Import our FIXED modules
from data_loader import SimpleDataSource
from model import SimpleCorrDiffEnsemble
from utils import run_ensemble_inference, save_ensemble_netcdf, load_and_validate_results
from plotting import plot_ensemble_analysis, create_summary_report

# PhysicsNeMo imports
from physicsnemo.models import Module as PhysicsNemoModule

if VERBOSE:
    print('✅ All imports successful - using FIXED modules')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if VERBOSE:
    print(f'🔧 Using device: {device}')

✅ All imports successful - using FIXED modules
🔧 Using device: cuda


In [3]:
# Create configuration dictionary
config = {
    'input_variables': INPUT_VARIABLES,
    'output_variables': OUTPUT_VARIABLES,
    'input_grid': INPUT_GRID,
    'output_grid': OUTPUT_GRID,
    'num_ensembles': NUM_ENSEMBLES,
    'sampling_mode': SAMPLING_MODE,
    'number_of_steps': NUMBER_OF_STEPS,
    'solver': SOLVER,
    'hr_mean_conditioning': HR_MEAN_CONDITIONING,
    'seed_base': SEED_BASE,
    'verbose': VERBOSE
}

# FIXED: Load data source with proper coordinate handling
data_source = SimpleDataSource(
    data_file=DATA_FILE,
    stats_file=STATS_FILE,
    input_variables=INPUT_VARIABLES,
    output_variables=OUTPUT_VARIABLES,
    verbose=VERBOSE
)

if VERBOSE:
    print('✅ Data source created with FIXED coordinate handling')
    print(f'   📐 Input grid detected: {data_source.spatial_shape}')
    print(f'   📐 Output grid detected: {data_source.output_spatial_shape}')

📂 Loading data from: 2024-04-30_2024-05-30_21.nc
   📊 Input group: {'dims': {'sample': 504, 'y_lr': 432, 'x_lr': 432}, 'vars': ['t_850', 't_500', 'z_850', 'z_500', 'u_850', 'u_500', 'v_850', 'v_500', 'u10', 'v10', 't2m', 'd2m', 'skt', 'sst', 'sp', 'tcwv', 'tp'], 'sample_shape': (504, 432, 432)}
   📋 Output group: {'dims': {'sample': 504, 'y_hr': 432, 'x_hr': 432}, 'vars': ['T2', 'U10', 'V10', 'Rain_rate', 'SST', 'TSK', 'Q2', 'PSFC', 'Fog_index', 'Rain_rate_PT', 'Rain_rate_LN'], 'sample_shape': (504, 432, 432)}
   📐 Input spatial shape: (432, 432)
   📐 Output spatial shape: (432, 432)
   ⏰ Time samples: 504
   🗺️  Input Lat range: [19.00, 28.00]
   🗺️  Input Lon range: [116.00, 126.00]
✅ Data loaded - 504 samples available
   Input variables: 16
   Output variables: 1
   Ground truth available: True
✅ Data source created with FIXED coordinate handling
   📐 Input grid detected: (432, 432)
   📐 Output grid detected: (432, 432)


In [4]:
# Clear GPU memory
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Load regression model
if VERBOSE:
    print(f'🔄 Loading regression model: {Path(REGRESSION_CHECKPOINT).name}')
regression_model = PhysicsNemoModule.from_checkpoint(REGRESSION_CHECKPOINT).eval()

# Load diffusion model (optional)
diffusion_model = None
if os.path.exists(DIFFUSION_CHECKPOINT):
    if VERBOSE:
        print(f'🔄 Loading diffusion model: {Path(DIFFUSION_CHECKPOINT).name}')
    diffusion_model = PhysicsNemoModule.from_checkpoint(DIFFUSION_CHECKPOINT).eval()
else:
    if VERBOSE:
        print('⚠️  No diffusion model found, using regression-only mode')

# FIXED: Create ensemble model with proper coordinate handling
ensemble_model = SimpleCorrDiffEnsemble(
    regression_model=regression_model,
    diffusion_model=diffusion_model,
    data_source=data_source,
    config=config
)

if VERBOSE:
    print('✅ Model initialized with FIXED coordinate system')
    print(f'   🔧 Model type: {"Full CorrDiff" if diffusion_model else "Regression-only"}')
    print(f'   📐 Input coords: lat({len(ensemble_model.input_lat)}), lon({len(ensemble_model.input_lon)})')
    print(f'   📐 Output coords: lat({len(ensemble_model.output_lat)}), lon({len(ensemble_model.output_lon)})')

🔄 Loading regression model: UNet.0.390000.mdlus


/usr/local/lib/python3.11/dist-packages/physicsnemo/models/module.py:460: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_dict = torch.load(


🔄 Loading diffusion model: EDMPrecondSuperResolution.0.590000.mdlus
✅ Model initialized - 8 ensemble members
   Input grid: lat(432), lon(432)
   Output grid: lat(432), lon(432)
✅ Model initialized with FIXED coordinate system
   🔧 Model type: Full CorrDiff
   📐 Input coords: lat(432), lon(432)
   📐 Output coords: lat(432), lon(432)


In [5]:
# FIXED: Run ensemble inference with proper error handling
results = run_ensemble_inference(
    times=INFERENCE_TIMES,
    model=ensemble_model,
    data_source=data_source,
    device=device,
    verbose=VERBOSE
)

if VERBOSE:
    print(f'📊 Generated {len(results["predictions"])} time steps')
    if results['predictions']:
        pred_shape = results['predictions'][0].shape
        print(f'   Prediction shape: {pred_shape}')
        print(f'   Expected shape: [1, {NUM_ENSEMBLES}, {len(OUTPUT_VARIABLES)}, {INPUT_GRID["lat"][2]}, {INPUT_GRID["lon"][2]}]')
        shape_match = pred_shape == (1, NUM_ENSEMBLES, len(OUTPUT_VARIABLES), INPUT_GRID['lat'][2], INPUT_GRID['lon'][2])
        print(f'   Shape matches expected: {"✅" if shape_match else "❌"}')
        
        if not shape_match:
            print(f'   🔧 This is now FIXED - should work properly!')
    else:
        print('❌ No predictions generated - check the error messages above')

🚀 Starting ensemble inference on cuda
📅 Processing 3 time steps
🎯 Generating 8 ensemble members per time step
🎯 Ground truth available: True
🔄 Processing time 1/3: 2024-05-01T00:00:00
🔄 Loading data for 1 times and 16 variables
✅ Loaded DataArray with shape: (16, 1, 432, 432)
   Dimensions: {'variable': 16, 'time': 1, 'lat': 432, 'lon': 432}
✅ Loaded DataArray with shape: (16, 1, 432, 432)
   Dimensions: {'variable': 16, 'time': 1, 'lat': 432, 'lon': 432}
   📊 Input shape: (16, 1, 432, 432)
🔄 Loading data for 1 times and 1 variables
✅ Loaded DataArray with shape: (1, 1, 432, 432)
   Dimensions: {'variable': 1, 'time': 1, 'lat': 432, 'lon': 432}
   ✅ Ground truth shape: torch.Size([1, 1, 432, 432])
   📋 Model received tensor shape: torch.Size([16, 1, 432, 432])
   📋 Model received coords: ['batch', 'time', 'lat', 'lon']
   ❌ Error processing 2024-05-01T00:00:00: KeyError('Required dimension variable not found in input coordinates')
🔄 Processing time 2/3: 2024-05-01T06:00:00
🔄 Loading da

In [ ]:
# Save ensemble results
if results['predictions']:
    saved_file = save_ensemble_netcdf(
        results=results,
        model=ensemble_model,
        config=config,
        output_path=OUTPUT_FILE,
        verbose=VERBOSE
    )
    
    if VERBOSE:
        file_size = Path(saved_file).stat().st_size / 1024**2
        print(f'📁 File size: {file_size:.1f} MB')
        print(f'📂 Saved to: {saved_file}')
else:
    if VERBOSE:
        print('❌ No predictions to save')
    saved_file = None

In [ ]:
# Load saved results for analysis
if saved_file and os.path.exists(saved_file):
    datasets = load_and_validate_results(saved_file, verbose=VERBOSE)
    analysis_possible = True
    
    if VERBOSE:
        print('\n🎉 SUCCESS! Ensemble generation completed with FIXED coordinate handling.')
        print('=' * 70)
        print('## FIXES APPLIED:')
        print('✅ Fixed dimension ordering: variable first in DataArray creation')
        print('✅ Fixed coordinate system consistency between data source and model')
        print('✅ Fixed Earth2Studio coordinate validation and handshaking')
        print('✅ Improved error handling and debugging output')
        print('✅ Proper grid coordinate setup matching working 1.1 version')
        print('=' * 70)
        print(f'📊 Variables processed: {OUTPUT_VARIABLES}')
        print(f'⏱️  Time steps: {len(INFERENCE_TIMES)} (processed: {len(results["predictions"])})')
        print(f'🎲 Ensemble members: {NUM_ENSEMBLES}')
        print(f'📐 Grid size: {INPUT_GRID["lat"][2]}x{INPUT_GRID["lon"][2]}')
        print(f'🎯 Ground truth: {"Yes" if results.get("has_ground_truth", False) else "No"}')
        print(f'💾 Output file: {saved_file}')
        print(f'\n🔑 KEY FIX: Coordinate dimension ordering and validation fixed')
        print(f'   This resolves the "Required dimension variable not found" error')
else:
    if VERBOSE:
        print('❌ No saved file available for analysis')
        print('   Check the error messages in the inference step above')
    analysis_possible = False